# 🛰️ RescueNet — Segmentation Model Training

**Model:** U-Net (ResNet-50 encoder, ImageNet weights)  
**Dataset:** RescueNet — 11-class disaster segmentation  
**Epochs:** 10  
**Output:** `/content/drive/MyDrive/DEEP_LEARNING/Pipeline-2/app/models/rescuenet_10epoch.pth`

> ⚠️ **IMPORTANT:** Go to `Runtime → Change runtime type → GPU` before running.

---

## Cell 1 — Check GPU

In [1]:
# ── Cell 1: GPU Verification ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise EnvironmentError(
        "❌ No GPU detected! "
        "Go to Runtime → Change runtime type → Hardware accelerator → GPU, then re-run."
    )
print(result.stdout)
print("✅ GPU is available — ready to train.")

Thu Apr  2 06:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Install Dependencies

In [2]:
# ── Cell 2: Install Required Packages ────────────────────────────────────────
# segmentation-models-pytorch brings timm + torchvision;
# albumentations is pinned to avoid numpy-2 breakage on current Colab.
!pip install -q \
    segmentation-models-pytorch \
    "albumentations>=1.3.0,<2.0.0" \
    opencv-python-headless

print("✅ All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 26.4 MB/s eta 0:00:00
✅ All packages installed.


## Cell 3 — Mount Google Drive

In [3]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted at /content/drive")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


## Cell 4 — Validate Dataset Paths

In [5]:
# ── Cell 4: Path Validation ───────────────────────────────────────────────────
import os

DATASET_ROOT  = "/content/drive/MyDrive/DEEP_LEARNING/Pipeline-1/dataset/train_set"
IMAGES_DIR    = os.path.join(DATASET_ROOT, "images")
MASKS_DIR     = os.path.join(DATASET_ROOT, "masks")

OUTPUT_DIR    = "/content/drive/MyDrive/DEEP_LEARNING/Pipeline-2/app/models"
OUTPUT_MODEL  = os.path.join(OUTPUT_DIR, "rescuenet_10epoch.pth")

assert os.path.isdir(IMAGES_DIR), f"❌ Images folder not found: {IMAGES_DIR}"
assert os.path.isdir(MASKS_DIR),  f"❌ Masks folder not found:  {MASKS_DIR}"

image_files = sorted(os.listdir(IMAGES_DIR))
mask_files  = sorted(os.listdir(MASKS_DIR))

print(f"📂 Images : {IMAGES_DIR}")
print(f"📂 Masks  : {MASKS_DIR}")
print(f"🖼️  Total image files : {len(image_files)}")
print(f"🎭 Total mask  files  : {len(mask_files)}")

# Ensure 1-to-1 correspondence by filename
image_names = set(image_files)
mask_names  = set(mask_files)
matched = image_names & mask_names
print(f"✅ Matched image-mask pairs: {len(matched)}")

if len(matched) == 0:
    raise ValueError(
        "No matching image-mask pairs found.\n"
        "Make sure each image filename exactly matches its mask filename."
    )

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n💾 Model will be saved to: {OUTPUT_MODEL}")

AssertionError: ❌ Images folder not found: /content/drive/MyDrive/DEEP_LEARNING/Pipeline-1/dataset/train_set/images

## Cell 5 — Imports

In [ ]:
# ── Cell 5: Imports ───────────────────────────────────────────────────────────
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

print(f"PyTorch  version : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"SMP      version : {smp.__version__}")

## Cell 6 — Dataset Class

In [ ]:
# ── Cell 6: RescueNet Dataset ─────────────────────────────────────────────────
class RescueNetDataset(Dataset):
    """
    Loads matched image-mask pairs.

    - Images : BGR → RGB, resized to 512×512, normalised to [0,1], float32 tensor (C,H,W)
    - Masks  : Grayscale, resized with INTER_NEAREST, LongTensor (H,W) with class indices 0-10
    """

    def __init__(self, images_dir: str, masks_dir: str, img_size: int = 512):
        self.images_dir = images_dir
        self.masks_dir  = masks_dir
        self.img_size   = img_size

        all_images = set(os.listdir(images_dir))
        all_masks  = set(os.listdir(masks_dir))
        # Only keep files present in both folders
        self.filenames = sorted(all_images & all_masks)

        if len(self.filenames) == 0:
            raise ValueError(
                f"No matching filenames found between:\n  {images_dir}\n  {masks_dir}"
            )

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]

        # ── Image ──────────────────────────────────────────────────────────────
        img_path = os.path.join(self.images_dir, fname)
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)           # BGR uint8
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)            # RGB uint8
        image = cv2.resize(image, (self.img_size, self.img_size), # 512×512
                           interpolation=cv2.INTER_LINEAR)
        image = image.astype(np.float32) / 255.0                  # [0,1]
        image = np.transpose(image, (2, 0, 1))                    # (C,H,W)
        image = torch.from_numpy(image)                           # FloatTensor

        # ── Mask ───────────────────────────────────────────────────────────────
        mask_path = os.path.join(self.masks_dir, fname)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)        # uint8 class idx
        if mask is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")
        mask = cv2.resize(mask, (self.img_size, self.img_size),   # 512×512
                          interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask).long()                      # LongTensor (H,W)

        return image, mask


# Quick sanity check ────────────────────────────────────────────────────────────
dataset = RescueNetDataset(IMAGES_DIR, MASKS_DIR, img_size=512)
print(f"✅ Dataset loaded — {len(dataset)} samples")

sample_img, sample_mask = dataset[0]
print(f"   Image tensor : {sample_img.shape}  dtype={sample_img.dtype}  "
      f"range=[{sample_img.min():.3f}, {sample_img.max():.3f}]")
print(f"   Mask  tensor : {sample_mask.shape}  dtype={sample_mask.dtype}  "
      f"unique classes={sample_mask.unique().tolist()}")

## Cell 7 — DataLoader

In [ ]:
# ── Cell 7: DataLoader ────────────────────────────────────────────────────────
BATCH_SIZE = 4
NUM_WORKERS = 2   # Colab supports 2 workers on most runtimes

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,    # Speeds up GPU transfers
    drop_last=False,    # Keep all samples
)

print(f"✅ DataLoader ready")
print(f"   Batch size : {BATCH_SIZE}")
print(f"   Batches    : {len(train_loader)}")
print(f"   Samples    : {len(dataset)}")

## Cell 8 — Build Model

In [ ]:
# ── Cell 8: Build U-Net Model ─────────────────────────────────────────────────
NUM_CLASSES = 11
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {DEVICE}")

model = smp.Unet(
    encoder_name    = "resnet50",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = NUM_CLASSES,
    activation      = None,        # Raw logits → CrossEntropyLoss handles softmax internally
)

model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ U-Net (ResNet-50) built")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")

## Cell 9 — Optimizer & Loss

In [ ]:
# ── Cell 9: Optimizer & Loss ──────────────────────────────────────────────────
LEARNING_RATE = 1e-4
EPOCHS        = 10

optimizer  = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion  = nn.CrossEntropyLoss()   # expects logits (N,C,H,W) + targets (N,H,W) LongTensor

print(f"✅ Optimizer : Adam  (lr={LEARNING_RATE})")
print(f"✅ Loss      : CrossEntropyLoss")
print(f"✅ Epochs    : {EPOCHS}")

## Cell 10 — Training Loop

In [ ]:
# ── Cell 10: Training Loop ────────────────────────────────────────────────────
epoch_losses = []

print("=" * 60)
print(" Starting RescueNet Training")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)   # (N, 3, 512, 512) float32
        masks  = masks.to(DEVICE, non_blocking=True)    # (N, 512, 512)    int64

        # ── Forward pass ───────────────────────────────────────────────────
        optimizer.zero_grad()
        logits = model(images)                          # (N, 11, 512, 512)

        # ── Loss ────────────────────────────────────────────────────────────
        loss = criterion(logits, masks)

        # ── Backward + Step ─────────────────────────────────────────────────
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Progress within epoch
        if (batch_idx + 1) % max(1, len(train_loader) // 5) == 0:
            print(f"  Epoch [{epoch:02d}/{EPOCHS}]  "
                  f"Batch [{batch_idx+1:04d}/{len(train_loader)}]  "
                  f"Loss: {loss.item():.4f}")

    # ── Epoch summary ───────────────────────────────────────────────────────
    avg_loss = running_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    print("-" * 60)
    print(f"  ✅ Epoch [{epoch:02d}/{EPOCHS}]  Avg Loss: {avg_loss:.4f}")
    print("-" * 60)

print("=" * 60)
print(" Training Complete!")
print("=" * 60)

## Cell 11 — Save Model

In [ ]:
# ── Cell 11: Save Model ───────────────────────────────────────────────────────
# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save full model state dict
torch.save(model.state_dict(), OUTPUT_MODEL)

# Verify the file was written
file_size_mb = os.path.getsize(OUTPUT_MODEL) / (1024 ** 2)
print(f"✅ Model saved successfully!")
print(f"   Path  : {OUTPUT_MODEL}")
print(f"   Size  : {file_size_mb:.1f} MB")

## Cell 12 — Loss Curve

In [ ]:
# ── Cell 12: Training Loss Curve ─────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses, marker='o', linewidth=2,
         color='#e74c3c', label='Train Loss')
plt.fill_between(range(1, EPOCHS + 1), epoch_losses, alpha=0.15, color='#e74c3c')
plt.title('RescueNet Training Loss (U-Net, ResNet-50)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.xticks(range(1, EPOCHS + 1))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nLoss per epoch:")
for i, l in enumerate(epoch_losses, 1):
    print(f"  Epoch {i:02d}: {l:.4f}")

## Cell 13 — Inference Sanity Check (Optional)

In [ ]:
# ── Cell 13: Inference Sanity Check ──────────────────────────────────────────
# RescueNet class palette (index → RGB colour for display)
PALETTE = np.array([
    [0,   0,   0  ],  # 0  Background
    [61,  230, 250],  # 1  Water
    [180, 120, 120],  # 2  Building — No Damage
    [235, 255, 7  ],  # 3  Building — Minor Damage
    [255, 184, 6  ],  # 4  Building — Major Damage
    [255, 0,   0  ],  # 5  Building — Total Destruction
    [34,  139, 34 ],  # 6  Vehicle
    [0,   0,   142],  # 7  Road
    [70,  70,  70 ],  # 8  Tree
    [152, 251, 152],  # 9  Pool
    [128, 0,   128],  # 10 Sand / Dirt
], dtype=np.uint8)

model.eval()

# Pick a random sample from the dataset
sample_idx  = 0
sample_img, sample_mask = dataset[sample_idx]
input_tensor = sample_img.unsqueeze(0).to(DEVICE)  # (1,3,512,512)

with torch.no_grad():
    logits = model(input_tensor)                    # (1, 11, 512, 512)
    pred   = logits.argmax(dim=1).squeeze(0)        # (512, 512) int64

pred_np   = pred.cpu().numpy().astype(np.uint8)     # (512, 512)
gt_np     = sample_mask.numpy().astype(np.uint8)    # (512, 512)

pred_rgb  = PALETTE[pred_np]                        # (512, 512, 3)
gt_rgb    = PALETTE[gt_np]                          # (512, 512, 3)

orig_img_np = sample_img.numpy().transpose(1, 2, 0) # (512, 512, 3) float

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(orig_img_np)
axes[0].set_title('Input Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(gt_rgb)
axes[1].set_title('Ground Truth Mask', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(pred_rgb)
axes[2].set_title('Predicted Mask (After 10 Epochs)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('RescueNet Inference Sanity Check', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"✅ Inference done. Unique predicted classes: {np.unique(pred_np).tolist()}")

## Cell 14 — Verify Saved Model (Reload Test)

In [ ]:
# ── Cell 14: Reload & Verify Saved Model ─────────────────────────────────────
# This confirms the .pth can be loaded exactly as the Flask app would load it.
verify_model = smp.Unet(
    encoder_name    = "resnet50",
    encoder_weights = None,        # No need to re-download imagenet; we load our weights
    in_channels     = 3,
    classes         = NUM_CLASSES,
    activation      = None,
)

state_dict = torch.load(OUTPUT_MODEL, map_location="cpu")
verify_model.load_state_dict(state_dict)
verify_model.eval()

# Quick forward pass to confirm shapes
dummy = torch.zeros(1, 3, 512, 512)
with torch.no_grad():
    out = verify_model(dummy)

assert out.shape == (1, NUM_CLASSES, 512, 512), f"Unexpected output shape: {out.shape}"

print("✅ Model reload verified!")
print(f"   Loaded from : {OUTPUT_MODEL}")
print(f"   Output shape: {tuple(out.shape)}  ← correct (N, 11, 512, 512)")
print("\n🎉 RescueNet training & export pipeline complete!")